In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

In [12]:
data = pd.read_csv("dataset.csv")

In [13]:
print(data.head(10))

   Marital status  Application mode  Application order  Course  \
0               1                 8                  5       2   
1               1                 6                  1      11   
2               1                 1                  5       5   
3               1                 8                  2      15   
4               2                12                  1       3   
5               2                12                  1      17   
6               1                 1                  1      12   
7               1                 9                  4      11   
8               1                 1                  3      10   
9               1                 1                  1      10   

   Daytime/evening attendance  Previous qualification  Nacionality  \
0                           1                       1            1   
1                           1                       1            1   
2                           1                       1          

### Assignment 1 (4 scores):

- Use Numpy only to construct the Logistic Regression model.
- Train that Logistic Regression model dataset using the Gradient Descent approach on the [Predict students’ dropout and academic success](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success) dataset. *Note that three class in this dataset must be merge into two class as: graduate and non-graduate (dropout or enroll)*.
- Evaluate that Logistic Regression model on the [Predict students’ dropout and academic success](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success) dataset.
- Visualize the loss function of the training process.

In [2]:
class LogisticRegression:
    def __init__(self, epochs: int, learning_rate: float):
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.losses = []
        self.metrics = []
        self.weights = None 

    def sigmoid(self, z: np.ndarray) -> np.ndarray:
        logits = (1 + np.exp(-z))
        logits = np.where(logits == 0, 10e-5, logits)
        return 1 / logits

    def compute_loss(self, y: np.ndarray, y_pred: np.ndarray) -> float:
        return -(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred)).mean()
    
    def fit(self, X: np.ndarray, y: np.ndarray, batch_size: int) -> None:
        n_samples = X.shape[0]
        n_features = X.shape[1]
        self.weights = np.zeros(n_features)

        for epoch in tqdm(range(self.epochs)):
            indices = np.arange(n_samples)
            np.random.shuffle(indices)

            X_shuffled = X[indices]
            y_shuffled = y[indices]

            for i in range(0, n_samples, batch_size):
                X_batch = X_shuffled[i: i + batch_size]
                y_batch = y_shuffled[i: i + batch_size]

                current_batch_size = X_batch.shape[0]

                linear_model = np.matmul(X_batch, self.weights)
                y_pred = self.sigmoid(linear_model)

                mini_batch_grad = (1 / current_batch_size) * np.matmul(X_batch.T, (y_pred - y_batch))
                self.weights -= self.learning_rate * mini_batch_grad

            y_pred_full = self.sigmoid(np.matmul(X, self.weights))
            loss = self.compute_loss(y, y_pred_full)
            self.losses.append(loss)

    def predict(self, X: np.ndarray, threshold: float) -> np.ndarray:
        linear_model = np.matmul(X, self.weights)
        y_pred = self.sigmoid(linear_model)
        return (y_pred >= threshold).astype(int)
        